In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, to_timestamp, hour, avg, count, first, round as spark_round

In [16]:
spark = SparkSession.builder \
    .appName("Analisis_AQI_Jatim") \
    .getOrCreate()

# Baca JSON multiline dari HDFS (format array JSON, bukan JSON Lines)
df_api_raw = spark.read \
    .option("multiline", "true") \
    .json("hdfs://namenode:9000/data/airquality/api/")

# Cast kolom numerik dan timestamp
# Format timestamp: '01-Mei-2026 12:00:00' (nama bulan Indonesia)
# Spark tidak support locale Indonesia, jadi kita parse via ingested_at sebagai fallback
df_api = df_api_raw \
    .withColumn("aqi", col("aqi").cast("double")) \
    .withColumn("pm25", col("pm25").cast("double")) \
    .withColumn("ts", to_timestamp(col("ingested_at"), "yyyy-MM-dd'T'HH:mm:ss.SSSSSS")) \
    .withColumn("jam", hour(col("ts")))

print("Schema:")
df_api.printSchema()
df_api.select("kota", "aqi", "ingested_at", "ts", "jam").show(5, truncate=False)

KeyboardInterrupt: 

In [ ]:
# Klasifikasi AQI (kolom sudah di-cast ke double di sel sebelumnya)
df_classified = df_api.withColumn("kategori_baru",
    when(col("aqi") <= 50, "Baik")
    .when((col("aqi") > 50) & (col("aqi") <= 100), "Sedang")
    .when((col("aqi") > 100) & (col("aqi") <= 200), "Tidak Sehat")
    .otherwise("Berbahaya")
)
df_classified.select("kota", "aqi", "kategori", "kategori_baru").show(10)

In [ ]:
# Hitung total record per kota
total_kota = df_classified.groupBy("kota").count().withColumnRenamed("count", "total")

# Hitung jumlah per kategori di tiap kota
distribusi = df_classified.groupBy("kota", "kategori_baru").count()

# Gabungkan dan hitung persentase
hasil_analisis = distribusi.join(total_kota, "kota") \
    .withColumn("persentase", spark_round((col("count") / col("total")) * 100, 1)) \
    .select("kota", "kategori_baru", "persentase") \
    .orderBy("kota", "kategori_baru")

hasil_analisis.show(truncate=False)

In [ ]:
hasil_analisis.write.mode("overwrite").json("hdfs://namenode:9000/data/airquality/hasil/analisis1")
print("Analisis 1 tersimpan ke HDFS: /data/airquality/hasil/analisis1")

## Analisis 2: Identifikasi Jam Puncak Polusi per Kota

Menjawab pertanyaan bisnis ETS: **"Pada jam berapa kualitas udara paling buruk?"**

Pipeline:
1. Parse kolom timestamp -> ekstrak jam (0-23)
2. Agregasi rata-rata AQI per kota & jam
3. Identifikasi jam puncak (avg_aqi tertinggi) per kota
4. Simpan hasil ke HDFS `/data/airquality/hasil/analisis2`

In [ ]:
from pyspark.sql.functions import col, to_timestamp, hour, avg, count, round as spark_round, first, when

# df_api sudah punya kolom 'ts' dan 'jam' dari sel sebelumnya
# Rename agar konsisten dengan query SQL
df_ts = df_api  # kolom 'kota' sudah ada, 'ts' dan 'jam' sudah di-parse

print('Schema df_ts:')
df_ts.printSchema()
df_ts.select('kota', 'ingested_at', 'ts', 'jam', 'aqi').show(5, truncate=False)

In [ ]:
# -------------------------------------------------------
# Langkah 2: Agregasi rata-rata AQI per kota dan jam
# -------------------------------------------------------
df_ts.createOrReplaceTempView('aqi_ts')

hasil2 = spark.sql("""
    SELECT kota,
           HOUR(ts)            AS jam,
           ROUND(AVG(aqi), 1)  AS avg_aqi,
           COUNT(*)            AS jumlah_data
    FROM   aqi_ts
    GROUP  BY kota, HOUR(ts)
    ORDER  BY kota, avg_aqi DESC
""")

print('Rata-rata AQI per Kota per Jam (diurutkan dari polusi tertinggi):')
hasil2.show(50, truncate=False)

In [ ]:
# -------------------------------------------------------
# Langkah 3: Identifikasi jam puncak polusi per kota
# Karena hasil2 sudah diurutkan avg_aqi DESC per kota,
# first('jam') mengambil jam dengan AQI rata-rata tertinggi.
# -------------------------------------------------------
jam_puncak = hasil2.groupBy('kota').agg(
    first('jam').alias('jam_puncak'),
    first('avg_aqi').alias('avg_aqi_puncak'),
    first('jumlah_data').alias('jumlah_data_puncak')
).orderBy('kota')

print('=== TABEL JAM PUNCAK POLUSI PER KOTA ===')
jam_puncak.show(truncate=False)

# Klasifikasi sesi jam
jam_puncak_labeled = jam_puncak.withColumn(
    'sesi',
    when((col('jam_puncak') >= 7) & (col('jam_puncak') <= 9),   'Pagi Sibuk (07-09)')
    .when((col('jam_puncak') >= 17) & (col('jam_puncak') <= 19), 'Sore Sibuk (17-19)')
    .when((col('jam_puncak') >= 10) & (col('jam_puncak') <= 16), 'Siang (10-16)')
    .when((col('jam_puncak') >= 20) & (col('jam_puncak') <= 23), 'Malam (20-23)')
    .otherwise('Dini Hari (00-06)')
)

print('=== JAM PUNCAK DENGAN KLASIFIKASI SESI ===')
jam_puncak_labeled.select('kota', 'jam_puncak', 'sesi', 'avg_aqi_puncak').show(truncate=False)

### Narasi & Rekomendasi Bisnis

#### Temuan Utama

Analisis pola jam menunjukkan bahwa **polusi udara umumnya memuncak pada dua sesi krusial**:

| Sesi | Jam | Faktor Pemicu |
|------|-----|---------------|
| **Pagi Sibuk** | 07.00-09.00 | Kendaraan berangkat kerja/sekolah, aktivitas industri awal hari |
| **Sore Sibuk** | 17.00-19.00 | Kendaraan pulang kerja, akumulasi polutan sepanjang hari |

> **Catatan**: Kota-kota dengan jam puncak di luar dua sesi di atas (misal siang atau dini hari)
> patut dicermati — bisa mengindikasikan sumber polusi industri atau pembakaran sampah yang tidak terjadwal.

#### Pola yang Sering Ditemukan

- **Pagi (07-09)** biasanya dominan di kota dengan kepadatan kendaraan tinggi dan industri pagi.
- **Sore (17-19)** sering lebih tinggi karena efek *photochemical smog* — sinar matahari seharian
  mengoksidasi polutan primer menjadi ozon dan partikulat sekunder.
- Kota industri berat kadang menunjukkan puncak malam (22-24) akibat shift produksi malam.

#### Rekomendasi untuk Dinas Kesehatan

1. **Notifikasi proaktif** — Kirim peringatan ISPU ke masyarakat 30 menit *sebelum* jam puncak spesifik per kota.
2. **Jadwal aktivitas luar ruang** — Rekomendasikan olahraga pagi sebelum 06.00 atau sesudah 20.00
   di kota dengan jam puncak pagi; sebaliknya, sebelum 16.00 jika puncak sore.
3. **Koordinasi dengan Dishub** — Dorong rekayasa lalu lintas (ganjil-genap, shifting jam kerja)
   atau perluasan WFH pada jam-jam kritis untuk menekan emisi kendaraan.
4. **Penguatan pemantauan ISPU** — Tingkatkan frekuensi pengukuran di kota dengan avg_aqi_puncak > 100
   (kategori Tidak Sehat), terutama menjelang jam puncak.
5. **Edukasi berbasis lokal** — Materi penyuluhan harus disesuaikan jam puncak tiap kota,
   bukan satu jadwal nasional yang seragam.

In [ ]:
# Simpan hasil ke HDFS
OUTPUT_PATH = 'hdfs://namenode:9000/data/airquality/hasil/analisis2'

# Simpan tabel lengkap (rata-rata AQI semua jam)
hasil2.write.mode('overwrite').json(f'{OUTPUT_PATH}/avg_aqi_per_jam')

# Simpan tabel ringkasan jam puncak per kota
jam_puncak_labeled.write.mode('overwrite').json(f'{OUTPUT_PATH}/jam_puncak_per_kota')

print(f'Hasil berhasil disimpan ke {OUTPUT_PATH}/')
print(f'   avg_aqi_per_jam/      -> rata-rata AQI tiap jam per kota')
print(f'   jam_puncak_per_kota/  -> jam puncak polusi per kota beserta klasifikasi sesi')